# 00 — Source Active eBay Listings (Browse API)

Scrapes ~5,000 active eBay GB listings across 5 popular second-hand categories
using the eBay Browse API (application token — no seller OAuth required).

**Output**
- 
- 
-  — one file per category

**Requirements** — add to :


In [ ]:
%pip install pandas pyarrow python-dotenv httpx --quiet

In [ ]:
import base64
import os
import time
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

In [ ]:
EBAY_CLIENT_ID     = os.getenv("EBAY_CLIENT_ID", "")
EBAY_CLIENT_SECRET = os.getenv("EBAY_CLIENT_SECRET", "")
EBAY_ENV           = os.getenv("EBAY_ENV", "production")
MARKETPLACE_ID     = "EBAY_GB"

_TOKEN_URLS = {
    "sandbox":    "https://api.sandbox.ebay.com/identity/v1/oauth2/token",
    "production": "https://api.ebay.com/identity/v1/oauth2/token",
}
_BROWSE_BASES = {
    "sandbox":    "https://api.sandbox.ebay.com/buy/browse/v1",
    "production": "https://api.ebay.com/buy/browse/v1",
}

TOKEN_URL    = _TOKEN_URLS[EBAY_ENV]
BROWSE_BASE  = _BROWSE_BASES[EBAY_ENV]
DATASETS_DIR = Path("../datasets")
DATASETS_DIR.mkdir(exist_ok=True)

assert EBAY_CLIENT_ID,     "EBAY_CLIENT_ID missing"
assert EBAY_CLIENT_SECRET, "EBAY_CLIENT_SECRET missing"

print(f"eBay env    : {EBAY_ENV}")
print(f"Marketplace : {MARKETPLACE_ID}")
print(f"Datasets dir: {DATASETS_DIR.resolve()}")

## App token (client credentials)

In [ ]:
def get_app_token() -> str:
    creds = f"{EBAY_CLIENT_ID}:{EBAY_CLIENT_SECRET}"
    basic = base64.b64encode(creds.encode()).decode()
    r = httpx.post(
        TOKEN_URL,
        headers={
            "Authorization": f"Basic {basic}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        data={
            "grant_type": "client_credentials",
            "scope": "https://api.ebay.com/oauth/api_scope",
        },
        timeout=20,
    )
    r.raise_for_status()
    return r.json()["access_token"]


APP_TOKEN = get_app_token()
print("App token acquired")

## Browse API helpers

In [ ]:
def search_page(
    token: str,
    query: str,
    offset: int = 0,
    limit: int = 200,
) -> list[dict]:
    r = httpx.get(
        f"{BROWSE_BASE}/item_summary/search",
        headers={
            "Authorization": f"Bearer {token}",
            "X-EBAY-C-MARKETPLACE-ID": MARKETPLACE_ID,
        },
        params={
            "q":      query,
            "limit":  min(limit, 200),
            "offset": offset,
            "sort":   "newlyListed",
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json().get("itemSummaries", [])


def flatten_item(item: dict, category_slug: str) -> dict:
    price    = item.get("price") or {}
    seller   = item.get("seller") or {}
    location = item.get("itemLocation") or {}
    image    = item.get("image") or {}

    try:
        price_value = float(price.get("value") or 0)
    except (TypeError, ValueError):
        price_value = 0.0

    return {
        "item_id":               item.get("itemId", ""),
        "title":                 item.get("title", ""),
        "category":              category_slug,
        "price":                 price_value,
        "currency":              price.get("currency", ""),
        "condition":             item.get("condition", ""),
        "condition_id":          item.get("conditionId", ""),
        "buying_options":        "|".join(item.get("buyingOptions") or []),
        "seller_feedback_score": seller.get("feedbackScore"),
        "seller_feedback_pct":   seller.get("feedbackPercentage"),
        "location_country":      location.get("country", ""),
        "location_city":         location.get("city", ""),
        "image_url":             image.get("imageUrl", ""),
        "listing_url":           item.get("itemWebUrl", ""),
    }

## Categories

Five high-volume second-hand categories on eBay GB.

In [ ]:
CATEGORIES = [
    {"slug": "smartphones", "query": "smartphone mobile phone"},
    {"slug": "laptops",     "query": "laptop notebook computer"},
    {"slug": "trainers",    "query": "trainers sneakers"},
    {"slug": "video_games", "query": "PS5 Xbox Nintendo game"},
    {"slug": "clothing",    "query": "jacket jeans hoodie dress"},
]

ITEMS_PER_CATEGORY  = 1_000
PAGE_SIZE           = 200
SLEEP_BETWEEN_PAGES = 0.4

print(f"Target: {ITEMS_PER_CATEGORY} x {len(CATEGORIES)} = {ITEMS_PER_CATEGORY * len(CATEGORIES):,} rows")

## Scrape

In [ ]:
all_rows: list[dict] = []

for cat in CATEGORIES:
    slug  = cat["slug"]
    query = cat["query"]
    rows: list[dict] = []
    offset = 0

    print(f"
{chr(9472) * 55}")
    print(f"Category : {slug}")
    print(f"Query    : {query!r}")

    while len(rows) < ITEMS_PER_CATEGORY:
        try:
            items = search_page(APP_TOKEN, query, offset=offset, limit=PAGE_SIZE)
        except httpx.HTTPStatusError as exc:
            print(f"  HTTP {exc.response.status_code} at offset={offset} -- stopping")
            break

        if not items:
            print(f"  No more results at offset={offset}")
            break

        rows.extend(flatten_item(i, slug) for i in items)
        print(f"  offset={offset:>5}  page_size={len(items):>3}  running={len(rows):>4}")

        offset += PAGE_SIZE
        time.sleep(SLEEP_BETWEEN_PAGES)

    kept = rows[:ITEMS_PER_CATEGORY]
    all_rows.extend(kept)
    print(f"  collected {len(kept)} items")

print(f"
Total rows (before dedup): {len(all_rows):,}")

## Build DataFrame & save

In [ ]:
df = pd.DataFrame(all_rows)
df = df.drop_duplicates(subset=["item_id"])
df["price"]                = pd.to_numeric(df["price"],                errors="coerce")
df["seller_feedback_score"] = pd.to_numeric(df["seller_feedback_score"], errors="coerce")
df["seller_feedback_pct"]  = pd.to_numeric(df["seller_feedback_pct"],  errors="coerce")

print(f"Shape after dedup: {df.shape}")
df.head(3)

In [ ]:
# Combined CSV
csv_path = DATASETS_DIR / "ebay_active_listings.csv"
df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}  ({csv_path.stat().st_size / 1024:.0f} KB)")

# Combined Parquet
try:
    parquet_path = DATASETS_DIR / "ebay_active_listings.parquet"
    df.to_parquet(parquet_path, index=False)
    print(f"Saved: {parquet_path}  ({parquet_path.stat().st_size / 1024:.0f} KB)")
except Exception as exc:
    print(f"Parquet skipped ({exc})")

# Per-category CSVs
for slug, group in df.groupby("category"):
    path = DATASETS_DIR / f"ebay_{slug}.csv"
    group.to_csv(path, index=False)
    print(f"Saved: {path}  ({len(group)} rows)")

## Summary stats

In [ ]:
print("=== Row counts ===")
print(df["category"].value_counts().to_string())

print("
=== Price stats (GBP) ===")
print(df.groupby("category")["price"].describe().round(2).to_string())

print("
=== Condition distribution ===")
cond = df.groupby(["category", "condition"]).size().unstack(fill_value=0)
print(cond.to_string())

print("
=== Buying options ===")
print(df["buying_options"].value_counts().head(10).to_string())